<a href="https://colab.research.google.com/github/rahilkhan-acadmic/APAIML-GradedMiniProject/blob/develop/capstone/Phase6_Monitoring-DriftDetection-and-overnance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from pydantic import BaseModel, validator, ValidationError

# 1. Real-time Data Validation Schema
class DataValidator(BaseModel):
    age: int
    billed_amount: float
    approval_ratio: float
    chronic_flag: int

    @validator('age')
    def age_within_range(cls, v):
        if not (0 <= v <= 120):
            raise ValueError('Age must be between 0 and 120')
        return v

    @validator('billed_amount')
    def positive_billing(cls, v):
        if v < 0:
            raise ValueError('Billed amount cannot be negative')
        return v

    @validator('approval_ratio')
    def ratio_limit(cls, v):
        if not (0.0 <= v <= 1.0):
            raise ValueError('Approval ratio must be between 0 and 1')
        return v

# 2. Batch Drift Detection Logic
def check_feature_drift(reference_df, current_df, threshold=0.1):
    """
    Compares the mean of key features between the training
    reference and the current production window.
    """
    drift_report = {}
    features_to_track = ['billed_amount', 'approval_ratio', 'age']

    for feature in features_to_track:
        ref_mean = reference_df[feature].mean()
        curr_mean = current_df[feature].mean()
        drift = abs(curr_mean - ref_mean) / ref_mean

        drift_report[feature] = {
            "drift_score": round(drift, 4),
            "alert": drift > threshold
        }
    return drift_report
